In [33]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoModelForCausalLM

/home/misgna/crxt-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset

In [9]:
# load dataset
dataset = pd.read_csv('dataset/asap_1_creative.csv')

In [10]:
dataset.describe()

,id,novelty,richness,logic,originality,coherence,freshness,appropriateness,individuality,persuasiveness,avg_score,overall
count,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000,1799.000000
mean,906.603113,2.663702,2.883268,2.907727,2.595331,2.961089,2.650917,2.919400,2.836020,2.946637,2.818416,2.818416
std,521.589873,0.859916,0.899105,0.847256,0.834930,0.833555,0.827360,0.769739,0.829324,0.822675,0.695588,0.695588
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,451.000000,2.000000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,2.000000,2.440000,2.440000
50%,911.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,2.890000,2.890000
75%,1350.000000,3.000000,3.000000,3.000000,3.000000,4.000000,3.000000,3.000000,3.000000,3.000000,3.330000,3.330000
max,1799.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,4.780000,4.780000


## Split dataset into training, validation, and test

In [37]:
dataset['total'] = dataset['overall'].round()

conditions = [dataset['total'] < 3, dataset['total'] == 3, dataset['total'] > 3]
choices = [-1, 0, 1]
dataset['class'] = np.select(conditions, choices)

dataset[dataset['class'] == -1].describe()

,id,novelty,richness,logic,originality,coherence,freshness,appropriateness,individuality,persuasiveness,avg_score,overall,total,class
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.0
mean,968.530000,1.794000,1.940000,1.988000,1.796000,2.070000,1.816000,2.126000,2.010000,2.040000,1.951960,1.951960,1.816000,-1.0
std,510.192024,0.569397,0.661267,0.672876,0.595898,0.685595,0.588929,0.677526,0.662403,0.650574,0.449784,0.449784,0.387872,0.0
min,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.0
25%,547.000000,1.000000,2.000000,2.000000,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,1.670000,1.670000,2.000000,-1.0
50%,1112.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.110000,2.110000,2.000000,-1.0
75%,1359.000000,2.000000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,2.000000,2.330000,2.330000,2.000000,-1.0
max,1798.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,4.000000,4.000000,4.000000,2.440000,2.440000,2.000000,-1.0


In [30]:
train, test = train_test_split(dataset, train_size=0.30, random_state=42, stratify=dataset['class'])
train, dev = train_test_split(train, train_size=0.50, random_state=42, stratify=train['class'])

# Save files
train.to_csv("dataset/train.csv", index=False)
dev.to_csv("dataset/dev.csv", index=False)
test.to_csv("dataset/test.csv", index=False)

# LLM Models

In [32]:
# load models
MODEL_CHECKPOINTS: dict[str, str] = {
    "gpt": "openai/gpt-oss-20b",
    "gemma_7b": "google/gemma-7b-it",
    "llama3_1b": "meta-llama/Llama-3.2-1B-Instruct",
    "llama3_8b": "meta-llama/Llama-3.1-8B-Instruct",
    "llama3_70b": "meta-llama/Llama-3.3-70B-Instruct",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.1",
    "mistral_8x7b": "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "falcon3_1b": "tiiuae/Falcon3-1B-Instruct",
    "falcon3_3b": "tiiuae/Falcon3-3B-Instruct",
    "falcon3_7b": "tiiuae/Falcon3-7B-Instruct",
    "falcon3_10b": "tiiuae/Falcon3-10B-Instruct",
    "deepseek_7b": "deepseek-ai/deepseek-llm-7b-chat",
    "deepseek_67b": "deepseek-ai/deepseek-llm-67b-chat",
    "deepseek_moe_16b": "deepseek-ai/deepseek-moe-16b-chat",
    "phi3.5_mini": "microsoft/Phi-3.5-mini-instruct",
    "phi3.5_moe": "microsoft/Phi-3.5-MoE-instruct",
    "phi3_mini_4k": "microsoft/Phi-3-mini-4k-instruct",
    "phi3_mini_medium_128k": "microsoft/Phi-3-medium-128k-instruct",
    "phi3_small_8k": "microsoft/Phi-3-small-8k-instruct",
    "phi3_small_128k": "microsoft/Phi-3-small-128k-instruct",
    "qwen3_8b": "Qwen/Qwen3-8B",
}
def get_checkpoint(model_id: str) -> str:
    """Retrieves HuggingFace model checkpoint path by model alias."""
    if model_id not in MODEL_CHECKPOINTS:
        raise KeyError(
            f"Unknown model_id: '{model_id}'. Available models: {list(MODEL_CHECKPOINTS.keys())}"
        )
    return MODEL_CHECKPOINTS[model_id]

In [ ]:
def load_model_and_tokenizer(checkpoint: str):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    if torch.cuda.is_available():
        print("Running on GPU")
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint,
            device_map="balanced",
            torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        )
    else:
        print("Running on CPU")
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint,
            torch_dtype=torch.float32,
        )
        
    #print(f"Device: {model.hf_device_map}")

    return model, tokenizer

# Prompts

In [ ]:
def extract_creative_feature_prompt(essay):
    return f"""
    You are an expert in creative essay writing assessment.
        
    # Task:
    - To extract the features that makes a given essay a creative.
        
    # Instruction:
    - Ignore content related information.
    - Focus on creative elements exhibited in the essay.
    - A bulleted list of core creative elements from the essay.

    # Input:
    Essay: {essay}
    """
    
def extract_creative_feature_diff_prompt(sample_1, sample_2, dimension):
    return f"""
    You are an expert in creative essay writing assessment.
        
    # Your task:
    - To extract the difference between the high scored and low scored essays provided trait scores and feedback.
        
    # Instruction:
    - Ignore content related information.
    - Use the trait scores hint.
    - Structure the difference to the scoring dimensions.
    - A bulleted list of differences between the two essays.

    # Input:
    ## Essay 1: 
    Essay: {sample_1['essay']}
    {dimension} score: {sample_1[dimension]} / 5.0

    ## Essay 2:
    Essay: {sample_2['essay']}
    {dimension} score: {sample_2[dimension]} / 5.0
    """

# Baselines

In [4]:
# baselines

# Proposed Approach

Select samples for each trait/dimensions, but first correlation between the trait scores

In [34]:
# 1. Define trait columns
traits = [
    'novelty', 'richness', 'logic', 'originality', 
    'coherence', 'freshness', 'appropriateness', 
    'individuality', 'persuasiveness'
]

# 2. Compute correlation of traits against target class
class_correlations = dataset[traits].corrwith(dataset['class']).sort_values(ascending=False)

print("Correlation with target 'class':")
print(class_correlations)

Correlation with target 'class':
persuasiveness     0.754497
richness           0.749832
coherence          0.744641
logic              0.742589
novelty            0.740049
appropriateness    0.728350
freshness          0.723283
individuality      0.717793
originality        0.697877
dtype: float64


In [35]:
for target_class, name in [(-1, 'Non-Creative'), (0, 'Conventional'), (1, 'Creative')]:
    subset = dataset[dataset['class'] == target_class]
    corr_matrix = subset[traits].corr(method='spearman')
    
    print(f"=== Correlation Matrix for Class {target_class} ({name}) ===")
    print(corr_matrix.round(3))
    print("\n" + "="*50 + "\n")


=== Correlation Matrix for Class -1 (Non-Creative) ===
                 novelty  richness  logic  originality  coherence  freshness  \
novelty            1.000     0.521  0.436        0.486      0.434      0.518   
richness           0.521     1.000  0.504        0.457      0.424      0.518   
logic              0.436     0.504  1.000        0.391      0.526      0.428   
originality        0.486     0.457  0.391        1.000      0.428      0.558   
coherence          0.434     0.424  0.526        0.428      1.000      0.365   
freshness          0.518     0.518  0.428        0.558      0.365      1.000   
appropriateness    0.389     0.465  0.452        0.397      0.546      0.407   
individuality      0.387     0.470  0.359        0.432      0.332      0.432   
persuasiveness     0.451     0.490  0.461        0.414      0.522      0.450   

                 appropriateness  individuality  persuasiveness  
novelty                    0.389          0.387           0.451  
richness    

# Result

In [ ]:
# graphs